# S5: LoRA SFT of Tiny Aya En-Thinker on teacher trajectories

**Before running:** Accelerator = GPU T4 x2, Internet = On, secret `HF_TOKEN` attached,
datasets `agentic-distill-src` (code) and `agentic-distill-sft` (train_v2.jsonl) attached.

Set `STAGE` below:
- `"smoke"`: memory test on the longest example, then a 10-example overfit (loss must fall, stay finite). ~20-30 min.
- `"full"`: 3 epochs on all examples, adapter saved after each epoch to /kaggle/working/out/lora/epochN.


In [ ]:
STAGE = "smoke"   # "smoke" or "full"
import sys, subprocess, os, glob
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
SFT = glob.glob("/kaggle/input/**/train_v2.jsonl", recursive=True)
print("sft:", SFT)
assert SFT, "attach the private agentic-distill-sft dataset"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # value never printed
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
Z=$(find /kaggle/input -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); \
else D=$(dirname "$(find /kaggle/input -name pyproject.toml | head -1)"); cp -r "$D"/. /kaggle/working/repo/; fi
pip install -q -U transformers peft accelerate safetensors
python -c "import torch, transformers, peft; print('torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, 'gpus', torch.cuda.device_count())"
cd /kaggle/working/repo && PYTHONPATH=src python -m pytest -q tests/test_loss_positions.py 2>&1 | tail -2


In [ ]:
import subprocess, os
SFT_PATH = SFT[0]
env = {**os.environ, "PYTHONPATH": "src", "CUDA_VISIBLE_DEVICES": "0",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def run(args):
    print(" ".join(args), flush=True)
    p = subprocess.Popen(["python", "-m", "agentic_distill.train_lora", "--data", SFT_PATH] + args,
                         cwd="/kaggle/working/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line: print(line, end="", flush=True)
    assert p.wait() == 0, f"train_lora exited {p.returncode}"

if STAGE == "smoke":
    run(["--out", "/kaggle/working/out/memtest", "--memtest"])
    run(["--out", "/kaggle/working/out/overfit10", "--limit", "10", "--grad-accum", "1", "--epochs", "4"])
else:
    run(["--out", "/kaggle/working/out/lora", "--epochs", "3", "--resume"])


In [ ]:
import json, glob
for p in glob.glob("/kaggle/working/out/*/train_log.jsonl"):
    recs = [json.loads(l) for l in open(p)]
    print(p, "steps", len(recs), "first loss", recs[0]["loss"], "last loss", recs[-1]["loss"],
          "skipped", sum(r["skipped"] for r in recs), "peak_mem_gb", max(r["peak_mem_gb"] for r in recs))
